In [ ]:
import pandas as pd
import kagglehub

# MODEL_NAME = "Pulk17/Fake-News-Detection"
# FAKE_CLASS = 0
# TRUE_CLASS = 1

MODEL_NAME = "dhruvpal/fake-news-bert" 
FAKE_CLASS = 1
TRUE_CLASS = 0

def first_n_words(text, n=300):
    words = text.split()
    return ' '.join(words[:n])

# Download latest version
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
real = pd.read_csv(f"{path}/True.csv")
fake = pd.read_csv(f"{path}/Fake.csv")

fake["label"] = FAKE_CLASS   # Fake
real["label"] = TRUE_CLASS   # Real

df_concat = pd.concat([fake, real], ignore_index=True)
df_concat["text"] = df_concat["title"] + " " + df_concat["text"]
df_concat["text"] = df_concat["text"].map(first_n_words)
df_concat = df_concat[["text", "label"]]

df = df_concat.sample(frac=1, random_state=42).reset_index(drop=True)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.


In [2]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [4]:
import numpy as np

def _to_list_of_str(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()
    if isinstance(texts, str):
        return [texts]

    return ["" if t is None else str(t) for t in texts]

def tokenize(texts):
    texts = _to_list_of_str(texts)
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

def predict(texts):
    texts = _to_list_of_str(texts)
    with torch.no_grad():
        inputs = tokenize(texts)
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return probs.cpu().numpy()

@torch.inference_mode()
def predict_torch(texts):
    if isinstance(texts, np.ndarray):
        texts = texts.tolist()

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=256,
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    out = model(**enc)

    logits = out.logits if hasattr(out, "logits") else out[0]
    probs = torch.softmax(logits, dim=-1)

    return probs.detach().cpu().numpy()

def predict_tokens(tokens):
    with torch.no_grad():
        outputs = model(**tokens)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    return probs

In [5]:
import torch
import numpy as np
from tqdm import tqdm

def batch_encodings(test_encodings, first, last):
    batch_input_ids = test_encodings['input_ids'][first:last]
    batch_attention_mask = test_encodings['attention_mask'][first:last]
    # batch_token_type_ids = test_encodings['token_type_ids'][first:last]

    return {
        'input_ids': batch_input_ids,
        'attention_mask': batch_attention_mask,
        # 'token_type_ids': batch_token_type_ids
    }

TEST_N_BATCHES = 40

test_encodings = tokenize(test_texts)
test_encodings = {k: v.to(device) for k, v in test_encodings.items()}
batch_size = 128
test_labels = test_labels[:TEST_N_BATCHES * batch_size]

# Define a batch size for inference to prevent OutOfMemory errors
predictions = []

# Get the total number of test samples
num_samples = test_encodings['input_ids'].shape[0]

with torch.no_grad():
    # for i in tqdm(range(0, num_samples, batch_size)):
    for i in tqdm(range(0, batch_size * TEST_N_BATCHES, batch_size)):
        # Extract batch from test_encodings
        batch = batch_encodings(test_encodings, i, i + batch_size)

        # Perform inference on the batch
        outputs = model(**batch)
        batch_predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        predictions.extend(batch_predictions)

# Convert the list of predictions to a numpy array
predictions = np.array(predictions)

100%|██████████| 40/40 [00:07<00:00,  5.54it/s]


In [6]:
present_tokens = test_encodings['input_ids'] > 0
present_tokens = torch.sum(present_tokens, dim = -1, dtype = int)
# present_tokens = torch.tensor(sorted(present_tokens.tolist()))

print(torch.max(present_tokens))
print(torch.median(present_tokens))
print(torch.min(present_tokens))

tensor(256, device='cuda:0')
tensor(256, device='cuda:0')
tensor(14, device='cuda:0')


In [7]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(test_labels, predictions)
print(f"Test Accuracy: {accuracy:.4f}\n")
print(classification_report(test_labels, predictions, target_names=["Fake", "Real"]))

Test Accuracy: 0.9992

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      2415
        Real       1.00      1.00      1.00      2705

    accuracy                           1.00      5120
   macro avg       1.00      1.00      1.00      5120
weighted avg       1.00      1.00      1.00      5120



In [8]:
import shap

masker = shap.maskers.Text(
    tokenizer=tokenizer,
    mask_token=tokenizer.mask_token,
)

explainer = shap.Explainer(
    predict_torch,
    masker,
    output_names=['fake', 'real'],
    algorithm='partition',
)

In [9]:
def top_words(shap_values, n=1, sample_idx=0, class_idx=0, top_k=10):
    dfs = []

    for sample_idx in range(n):
        values = shap_values.values[sample_idx][:, class_idx]
        tokens = shap_values.data[sample_idx]

        df = pd.DataFrame({
            "token": tokens,
            "importance": values
        })

        df = df[df.token.str.strip() != ""]
        df["abs"] = df.importance.abs()

        dfs.append(df.sort_values("abs", ascending=False).head(top_k))

    return dfs

In [10]:
import matplotlib.pyplot as plt

def plot_top_words(dfs):
    for i in range(len(dfs)):
        plt.figure(figsize=(8, 4))
        plt.barh(
            dfs[i].token[::-1],
            dfs[i].importance[::-1]
        )
        plt.axvline(0, color="black", linewidth=0.8)
        plt.title("Most Important Words (SHAP)")
        plt.xlabel("Impact on prediction")
        plt.tight_layout()
        plt.show()

# plot_top_words(top_words(shap_values, 2))

In [ ]:
import numpy as np
import copy

N_TEXTS = 50
ROUNDS = 13          
MAX_STEPS = 50
TARGETED = True
TARGET_CLASS = FAKE_CLASS 
EPS = 1e-3

texts = None
if TARGET_CLASS == TRUE_CLASS:
    texts = df_concat.iloc[:int(2e4)].sample(frac=N_TEXTS/2e4, random_state=42).reset_index(drop=True)
    texts = texts["text"].dropna().tolist()
else:
    texts = df_concat.iloc[-int(2e4):].sample(frac=N_TEXTS/2e4, random_state=42).reset_index(drop=True)
    texts = texts["text"].dropna().tolist()

current_texts = copy.deepcopy(texts)
current_pred  = predict_torch(current_texts)
first_pred    = current_pred.copy()

def join_tokens(tokens):
    return "".join(tokens)

def current_target_class(i):
    return TARGET_CLASS if TARGETED else int(np.argmax(current_pred[i]))

def pick_ranking(shap_vals_i, class_idx, mode="pos"):
    v = shap_vals_i[:, class_idx]

    if mode == "abs":
        return np.argsort(-np.abs(v))
    else:
        return np.argsort(-v)

def compute_change(i, new_pred_i):
    if TARGETED:
        cls = TARGET_CLASS
        return new_pred_i[cls] - current_pred[i, cls]
    else:
        cls = int(np.argmax(current_pred[i]))
        return current_pred[i, cls] - new_pred_i[cls]

def check_flipped(new_pred_i):
    if TARGETED:
        cls = TARGET_CLASS
        return new_pred_i[cls] > new_pred_i[1-cls]

    else:
        return False

flipped = np.zeros(N_TEXTS, dtype=bool)

for round_num in range(ROUNDS):
    shap_values = explainer(current_texts)

    ranks = []
    toks_cache = []
    for i in range(N_TEXTS):
        original_class = 1 - TARGET_CLASS 
        ranks.append(pick_ranking(shap_values.values[i], original_class, mode="pos"))
        toks_cache.append(list(shap_values.data[i]))

    altered = np.zeros(N_TEXTS, dtype=bool)
    for step in range(MAX_STEPS):
        new_texts = current_texts.copy()

        for i in range(N_TEXTS):
            if flipped[i]:
                continue

            r = ranks[i]
            deleted_idx = int(r[step]) if step < len(r) else r[0]

            toks = toks_cache[i].copy()
            toks[deleted_idx] = ""
            new_texts[i] = join_tokens(toks)

        new_pred = predict_torch(new_texts)

        for i in range(N_TEXTS):
            if flipped[i]:
                continue

            flipped[i] = check_flipped(new_pred[i])
            if flipped[i]:
                print(f"Sample {i} flipped.")

            change = compute_change(i, new_pred[i])
            r = ranks[i]
            deleted_idx = int(r[step]) if step < len(r) else r[0]
            deleted_word = toks_cache[i][deleted_idx]

            if change > EPS:
                print(f"Deleting word {deleted_word} in sample {i} with attribution {shap_values.values[i][deleted_idx]} prediction change {change}")

                toks_cache[i][deleted_idx] = ""
                current_texts[i] = new_texts[i]
                current_pred[i]  = new_pred[i]
                altered[i] = True

    print("Altering unaltered samples")
    for i in range(N_TEXTS):
        if not altered[i] and not flipped[i]:
            deleted_idx = ranks[i][0]
            deleted_word = toks_cache[i][deleted_idx]

            # print(f"Deleting word {deleted_word} in sample {i} with attribution {shap_values.values[i][deleted_idx]}")

            toks = toks_cache[i].copy()
            toks[deleted_idx] = ""
            current_texts[i] = join_tokens(toks)

            toks_cache[i][deleted_idx] = ""
    
    current_pred = predict_torch(current_texts)

    flips = np.sum(np.argmax(current_pred, axis=1) != np.argmax(first_pred, axis=1))
    print(f"round={round_num:02d} flips={flips}/{N_TEXTS}")

PartitionExplainer explainer: 51it [00:52,  1.24s/it]                        


1.9007302e-07
2.9802777e-07
1.541639e-07
1.2323653e-09
-1.4547368e-08
5.8033493e-08
5.189031e-08
5.2051746e-08
1.9780146e-08
3.480818e-08
3.0449428e-08
4.8713446e-08
4.890444e-08
4.3224645e-08
4.2211013e-08
6.0108505e-08
1.3828867e-08
3.64048e-08
3.767991e-08
6.252776e-09
2.2278073e-09
9.34233e-09
-1.4388206e-09
5.9030754e-09
8.197276e-09
4.202593e-08
6.5665517e-09
3.8630787e-09
5.392758e-08
5.3987605e-09
4.7293724e-09
5.9267222e-09
4.8712536e-09
7.699327e-09
5.226866e-09
3.9241513e-08
3.614241e-08
4.169806e-08
3.6881374e-08
3.7620794e-08
4.254616e-09
4.4324224e-09
2.761226e-09
-1.6102604e-09
3.3557626e-08
3.69173e-08
5.3573785e-09
3.4724508e-09
5.1913958e-09
4.0779923e-08
Altering unaltered samples
round=00 flips=0/50


PartitionExplainer explainer: 51it [00:52,  1.25s/it]                        


1.3670387e-07
Sample 14 flipped.
Deleting word Reuters in sample 14 with attribution [ 0.4259192 -0.4259192] prediction change 0.9999576807022095
Sample 20 flipped.
Deleting word -  in sample 20 with attribution [ 0.42052236 -0.42052236] prediction change 0.9998348951339722
Deleting word Reuters in sample 25 with attribution [ 0.34714143 -0.34714143] prediction change 0.16153766214847565
Deleting word Reuters in sample 36 with attribution [ 0.10922659 -0.10922659] prediction change 0.0017474344931542873
1.8751653e-06
Sample 16 flipped.
Deleting word -  in sample 16 with attribution [ 0.28604322 -0.28604322] prediction change 0.9978375434875488
Sample 25 flipped.
Deleting word -  in sample 25 with attribution [ 0.30026978 -0.30026978] prediction change 0.8384339213371277
Sample 31 flipped.
Deleting word -  in sample 31 with attribution [ 0.20972459 -0.20972459] prediction change 0.9999622702598572
Sample 35 flipped.
Deleting word -  in sample 35 with attribution [ 0.28206261 -0.28206261

PartitionExplainer explainer: 51it [00:50,  1.20s/it]                        


...

Altering unaltered samples
round=10 flips=43/50


PartitionExplainer explainer: 51it [00:50,  1.24s/it]                        


Sample 49 flipped.
Deleting word . in sample 49 with attribution [ 0.00699148 -0.00699148] prediction change 0.9997614026069641
Altering unaltered samples
round=11 flips=44/50


PartitionExplainer explainer: 51it [00:51,  1.26s/it]                        


Sample 10 flipped.
Deleting word -  in sample 10 with attribution [ 0.12378478 -0.12378478] prediction change 0.9848964810371399
Sample 18 flipped.
Deleting word )  in sample 18 with attribution [ 0.09032131 -0.09032131] prediction change 0.9929264187812805
Sample 47 flipped.
Deleting word a  in sample 47 with attribution [ 0.02229269 -0.02229269] prediction change 0.8335051536560059
Altering unaltered samples
round=12 flips=47/50
